# Воспроизведение итогового обучения LightGBM

Это отдельный запуск, не требуется для просмотра проекта. Скачай исходный архив, выполни подготовку в credit_scoring.ipynb с RUN_RAW_ANALYSIS=True, затем установи TRAIN_NEW_MODEL=True ниже. По умолчанию обучение выключено.

Новый эксперимент записывается в experiments/ с отдельной датой. Опубликованная модель models/lightgbm остаётся прежней. Архив проекта содержит базовые таблицы, но при клонировании Git их нужно сформировать самостоятельно. Воспроизводим алгоритм и разбиение; совпадение весов между разными версиями окружения не гарантируется.

In [ ]:
# Необходимо установить requirements.txt из корня проекта

In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
                     if (p / 'requirements.txt').is_file() and (p / 'src').is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Открой Jupyter внутри credit-scoring.')
sys.path.insert(0, str(PROJECT_ROOT))
TRAIN_NEW_MODEL = False  # Включать только для нового обучения.
print('Новое обучение:', TRAIN_NEW_MODEL)

Новое обучение: False


In [ ]:
if TRAIN_NEW_MODEL:
    from pathlib import Path
    from datetime import datetime, timezone
    import json
    import gc
    import os
    import time
    import shutil
    import tempfile
    import platform
    import hashlib
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import pyarrow as pa
    import pyarrow.compute as pc
    import pyarrow.parquet as pq
    import lightgbm as lgb
    import joblib
    import sklearn
    from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, log_loss, RocCurveDisplay, PrecisionRecallDisplay
    from IPython.display import display

    ROOT_DIR = PROJECT_ROOT
    RAW_DATA_DIR = PROJECT_ROOT / 'data/raw/data_for_competition'
    BASE_DATA_DIR = PROJECT_ROOT / 'data/prepared'
    RANDOM_STATE = 42
    N_THREADS = 2  # Как в завершённом итоговом обучении
    N_SHARDS = 80
    BATCH_SIZE = 150_000
    NUM_BOOST_ROUND = 600
    CHECKPOINT_EVERY = 100
    APPROVAL_TARGET = 0.80
    MAX_TRAIN_ROWS = None  # None = все заявки прежнего train-периода; для короткой проверки можно 500_000.
    SAVE_PREPARED_SHARDS = False

    stamp = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
    RUN_DIR = ROOT_DIR / 'experiments' / 'final_lightgbm_full_period' / stamp
    MODEL_DIR = RUN_DIR / 'model'
    MODEL_DIR.mkdir(parents=True, exist_ok=True)

    LOCAL_DIR = Path(tempfile.mkdtemp(prefix='alfa_final_'))
    TRAIN_TEXT_PATH = LOCAL_DIR / 'train.tsv'
    print('Результаты:', RUN_DIR)
    print('Временные файлы:', LOCAL_DIR)
    print('Свободно локально, GB:', round(shutil.disk_usage(LOCAL_DIR).free / 2**30, 1))


## 1. Граница train и словари категорий

Считываем старые ID для фиксации границы, а не меняем прежнее разбиение. Сохраняем ID всех частей и их хеши.

Словарь частот оставляем прежним. Категории последнего продукта также кодируем по train: неизвестные значения получают -1. Категориальные номера передаются LightGBM с явным списком категориальных столбцов. Это не арифметический порядок случайных кодов.

Holdout-признаки и метки здесь не загружаем. Исходный CSV таргета читаем для отбора train, но в модель поступят только метки train.

In [ ]:
if TRAIN_NEW_MODEL:
    with open(BASE_DATA_DIR / 'feature_settings.json', encoding='utf-8') as file:
        feature_settings = json.load(file)
    categorical_features = feature_settings['categorical_features']
    base_numerical_features = feature_settings['numerical_features']
    base_feature_columns = base_numerical_features + categorical_features
    category_vocabulary = feature_settings['category_vocabulary']
    old_train = pd.read_parquet(BASE_DATA_DIR / 'train_features.parquet', columns=['id'] + categorical_features).set_index('id')
    valid_base = pd.read_parquet(BASE_DATA_DIR / 'validation_features.parquet').set_index('id')
    holdout_ids = pd.read_parquet(BASE_DATA_DIR / 'holdout_features.parquet', columns=['id']).id.to_numpy()
    assert old_train.index.is_unique and valid_base.index.is_unique
    train_boundary = int(old_train.index.max())
    assert train_boundary < valid_base.index.min() and valid_base.index.max() < holdout_ids.min()
    all_labels = pd.read_csv(RAW_DATA_DIR / 'train_target.csv').set_index('id').flag
    train_labels = all_labels.loc[all_labels.index <= train_boundary].sort_index().astype('int8')
    if MAX_TRAIN_ROWS is not None and MAX_TRAIN_ROWS < len(train_labels):
        selected = np.sort(np.random.default_rng(RANDOM_STATE).choice(train_labels.index.to_numpy(), MAX_TRAIN_ROWS, replace=False))
        train_labels = train_labels.loc[selected]
    train_ids = train_labels.index.to_numpy()
    valid_ids = valid_base.index.to_numpy()
    y_valid = valid_base.flag.astype(int)
    assert pd.Index(train_ids).intersection(valid_ids).empty
    assert pd.Index(train_ids).intersection(holdout_ids).empty
    category_maps = {
        col: {value: index for index, value in enumerate(sorted(set(old_train[col].dropna().astype(str)) | {'missing'}))}
        for col in categorical_features
    }
    for name, ids in [('train', train_ids), ('validation', valid_ids), ('holdout', holdout_ids)]:
        pd.DataFrame({'id': np.sort(ids)}).to_csv(RUN_DIR / f'{name}_ids.csv', index=False)
    manifest = {'train_rows': len(train_ids), 'train_boundary': train_boundary,
        'validation_rows': len(valid_ids), 'holdout_rows': len(holdout_ids), 'holdout_evaluated': False,
        'id_hashes': {name: hashlib.sha256(np.sort(np.asarray(ids, dtype='int64')).tobytes()).hexdigest()
                      for name, ids in [('train', train_ids), ('validation', valid_ids), ('holdout', holdout_ids)]},
        'target': feature_settings['target'], 'vocabulary_source': str(BASE_DATA_DIR), 'seed': RANDOM_STATE}
    with open(RUN_DIR / 'split_manifest.json', 'w', encoding='utf-8') as file:
        json.dump(manifest, file, ensure_ascii=False, indent=2)
    print('Train:', len(train_ids), '| доля дефолтов:', round(float(train_labels.mean()), 5))
    print('Validation:', len(valid_ids), '| доля дефолтов:', round(float(y_valid.mean()), 5))
    del old_train, all_labels
    gc.collect()

## 2. Те же расширенные признаки

Объём истории, разнообразие кодов, доли флагов, частоты категорий, последний продукт, изменения платёжных кодов; дополнительно count_0/count_1 флагов и признаки последних 3/5 продуктов по rn.

Суммы случайных кодов не считаем. Последние 3/5 продуктов не называем месяцами. rn нужен для порядка продуктов, id не входит в модель. Сохраняем код подготовки рядом с моделью для будущего применения.

In [ ]:
if TRAIN_NEW_MODEL:
    train_files = sorted((RAW_DATA_DIR / 'train_data').glob('*.pq'))
    if not train_files:
        raise FileNotFoundError(f'Нет исходных .pq в {RAW_DATA_DIR / "train_data"}; поправь RAW_DATA_DIR')
    raw_columns = pq.ParquetFile(train_files[0]).schema_arrow.names
    code_columns = [col for col in raw_columns if col.startswith(('pre_', 'enc_'))]
    flag_columns = [col for col in raw_columns if col.startswith('is_zero_') or col in ['pclose_flag', 'fclose_flag']]
    payment_columns = sorted([col for col in raw_columns if col.startswith('enc_paym_')],
                             key=lambda col: int(col.rsplit('_', 1)[1]))
    category_vocabulary = feature_settings['category_vocabulary']
    categorical_features = feature_settings['categorical_features']
    numerical_features = base_numerical_features
    feature_columns = numerical_features + categorical_features
    def make_base_features(raw, application_ids, vocabulary):
        groups = raw.groupby('id', sort=False)
        index = pd.Index(application_ids, name='id')
        features = pd.DataFrame(index=index)
        product_counts = groups.size()
        features['n_credit_products'] = product_counts.reindex(index, fill_value=0)
        features['has_history'] = (features.n_credit_products > 0).astype('int8')

        diversity = groups[code_columns].nunique().add_suffix('__nunique')
        missing_share = raw[code_columns].isna().groupby(raw.id).mean().add_suffix('__missing_share')
        # Среднее бинарного флага = доля единиц; отдельно учитываем пропуски флагов.
        flag_share = groups[flag_columns].mean().add_suffix('__share')
        flag_missing = raw[flag_columns].isna().groupby(raw.id).mean().add_suffix('__missing_share')
        features = features.join([diversity, missing_share, flag_share, flag_missing])

        frequency_blocks = []
        for col, categories in vocabulary.items():
            counts = raw.groupby(['id', col]).size().unstack(col, fill_value=0)
            counts = counts.reindex(columns=categories, fill_value=0)
            shares = counts.div(product_counts, axis=0).fillna(0)
            shares.columns = [f'{col}__code_{value}_share' for value in categories]
            frequency_blocks.append(shares)
            unknown = raw[col].notna() & ~raw[col].isin(categories)
            unknown_share = unknown.groupby(raw.id).mean().rename(f'{col}__unknown_share')
            frequency_blocks.append(unknown_share.to_frame())
        features = features.join(frequency_blocks)

        latest = raw.sort_values(['id', 'rn']).drop_duplicates('id', keep='last').set_index('id')
        for col in code_columns + flag_columns:
            values = latest[col].reindex(index)
            features[f'latest__{col}'] = values.map(
                lambda value: 'missing' if pd.isna(value) else f'code_{value:g}'
            )

        payments = raw[payment_columns]
        left = payments.iloc[:, :-1].to_numpy()
        right = payments.iloc[:, 1:].to_numpy()
        observed = pd.notna(left) & pd.notna(right)
        comparisons = observed.sum(axis=1)
        changes = ((left != right) & observed).sum(axis=1)
        change_share = np.divide(changes, comparisons,
                                 out=np.full(len(raw), np.nan), where=comparisons > 0)
        changes_by_id = pd.Series(change_share, index=raw.index).groupby(raw.id)
        features['payment_change_share_mean'] = changes_by_id.mean()
        features['payment_change_share_max'] = changes_by_id.max()
        features['payment_pair_observed_share'] = pd.Series(
            observed.mean(axis=1), index=raw.index).groupby(raw.id).mean()

        numeric_columns = features.select_dtypes(include='number').columns
        no_history = features.has_history.eq(0)
        features.loc[no_history, numeric_columns] = 0
        share_columns = [col for col in features if col.endswith('_share') and '__code_' in col]
        features[share_columns] = features[share_columns].fillna(0)
        features[numeric_columns] = features[numeric_columns].astype('float32')
        return features
    def make_application_features(raw, application_ids, vocabulary):
        features = make_base_features(raw, application_ids, vocabulary)
        count_features = {}
        for flag in flag_columns:
            for value in [0, 1]:
                count = raw[flag].eq(value).groupby(raw.id).sum()
                count_features[f'extra__{flag}__count_{value}'] = count.reindex(features.index, fill_value=0)
        features = features.join(pd.DataFrame(count_features, index=features.index))
        ordered = raw.sort_values(['id', 'rn'])
        diversity_fields = [col for col in ['enc_loans_credit_type', 'enc_loans_credit_status',
            'pre_loans_outstanding', 'pre_util', 'pre_loans3060', 'pre_loans6090', 'pre_loans90'] if col in raw]
        for k in [3, 5]:
            recent = ordered.groupby('id', sort=False).tail(k)
            recent_groups = recent.groupby('id', sort=False)
            shares = recent_groups[flag_columns].mean().add_prefix(f'extra__last_{k}__').add_suffix('__share')
            diversity = recent_groups[diversity_fields].nunique().add_prefix(f'extra__last_{k}__').add_suffix('__nunique')
            features = features.join([shares, diversity])
        extras = [col for col in features if col.startswith('extra__')]
        features.loc[features.has_history.eq(0), extras] = 0
        features[extras] = features[extras].astype('float32')
        return features

## 3. Переносим строки истории в локальные части

Берём только train и прежнюю validation. Все строки одной заявки попадают в одну часть по `id % N_SHARDS`, даже если исходная история лежит в разных `.pq`. Это предотвращает обрезание кредитной истории на границе файлов.

Затем по одной части строим признаки, кодируем категории и дописываем строки train в TSV. Validation маленькая и остаётся в памяти. Готовый train целиком в pandas не создаём.

In [ ]:
if TRAIN_NEW_MODEL:
    all_ids = np.sort(np.concatenate([train_ids, valid_ids]))
    wanted = pa.array(all_ids)
    writers = {}
    try:
        for path in train_files:
            kept = 0
            for batch in pq.ParquetFile(path).iter_batches(batch_size=BATCH_SIZE):
                table = pa.Table.from_batches([batch])
                table = table.filter(pc.is_in(table['id'], value_set=wanted))
                if not len(table):
                    continue
                buckets = table['id'].to_numpy() % N_SHARDS
                for shard in np.unique(buckets):
                    shard = int(shard)
                    piece = table.filter(pa.array(buckets == shard))
                    if shard not in writers:
                        writers[shard] = pq.ParquetWriter(LOCAL_DIR / f'raw_{shard}.parquet', piece.schema)
                    writers[shard].write_table(piece)
                kept += len(table)
            print(path.name, '| строк истории:', kept, flush=True)
    finally:
        for writer in writers.values():
            writer.close()
    del wanted, writers
    gc.collect()

In [ ]:
if TRAIN_NEW_MODEL:
    def encode_prepared(features, schema):
        # медиану не подбираем
        numeric = features[schema['numerical_features']].to_numpy(dtype='float32', copy=True)
        encoded = np.empty((len(features), len(schema['categorical_features'])), dtype='float32')
        for i, col in enumerate(schema['categorical_features']):
            values = features[col].fillna('missing').astype(str)
            encoded[:, i] = values.map(schema['category_maps'][col]).fillna(-1).to_numpy(dtype='float32')
        return np.concatenate([numeric, encoded], axis=1)

    valid_parts = []
    written_rows = 0
    schema = None
    if SAVE_PREPARED_SHARDS:
        (RUN_DIR / 'prepared_train').mkdir(exist_ok=True)
    for shard in range(N_SHARDS):
        ids = all_ids[all_ids % N_SHARDS == shard]
        if not len(ids):
            continue
        raw_path = LOCAL_DIR / f'raw_{shard}.parquet'
        raw = pd.read_parquet(raw_path) if raw_path.exists() else pd.DataFrame(columns=raw_columns)
        features = make_application_features(raw, ids, category_vocabulary)
        if schema is None:
            extra_columns = [col for col in features if col.startswith('extra__')]
            numerical_features = base_numerical_features + extra_columns
            feature_columns = numerical_features + categorical_features
            schema = {'feature_columns': feature_columns, 'numerical_features': numerical_features,
                      'categorical_features': categorical_features, 'category_maps': category_maps,
                      'categorical_indices': list(range(len(numerical_features), len(feature_columns))),
                      'numeric_missing': 'native LightGBM NaN', 'unknown_category': -1}
        part_train_ids = train_ids[train_ids % N_SHARDS == shard]
        if len(part_train_ids):
            train_part = features.loc[part_train_ids, feature_columns]
            matrix = encode_prepared(train_part, schema)
            export = pd.DataFrame(matrix, columns=feature_columns)
            export.insert(0, 'flag', train_labels.loc[part_train_ids].to_numpy())
            export.to_csv(TRAIN_TEXT_PATH, mode='a', header=written_rows == 0, index=False,
                          sep='\t', na_rep='nan', float_format='%.9g')
            written_rows += len(export)
            if SAVE_PREPARED_SHARDS:
                prepared = train_part.copy()
                prepared['flag'] = train_labels.loc[part_train_ids]
                prepared.reset_index().to_parquet(RUN_DIR / 'prepared_train' / f'part_{shard:03d}.parquet', index=False)
            del train_part, matrix, export
        part_valid_ids = valid_ids[valid_ids % N_SHARDS == shard]
        if len(part_valid_ids):
            valid_parts.append(features.loc[part_valid_ids, feature_columns].copy())
        del raw, features
        if raw_path.exists():
            raw_path.unlink()
        gc.collect()
        print(f'Признаки {shard + 1}/{N_SHARDS} | записано train: {written_rows}', flush=True)
    assert written_rows == len(train_ids)
    X_valid = pd.concat(valid_parts).loc[valid_ids, feature_columns]
    np.testing.assert_allclose(X_valid[base_numerical_features].to_numpy(), valid_base[base_numerical_features].to_numpy(),
                               equal_nan=True, rtol=1e-6)
    assert X_valid[categorical_features].equals(valid_base[categorical_features])
    validation_export = X_valid.copy()
    validation_export['flag'] = y_valid
    validation_export.reset_index().to_parquet(RUN_DIR / 'validation_features.parquet', index=False)
    with open(MODEL_DIR / 'schema.json', 'w', encoding='utf-8') as file:
        json.dump(schema, file, ensure_ascii=False, indent=2)
    feature_settings = {**feature_settings, 'numerical_features': numerical_features,
                        'train_rows': len(train_ids), 'extended_features': True, 'native_nan': True}
    with open(MODEL_DIR / 'feature_settings.json', 'w', encoding='utf-8') as file:
        json.dump(feature_settings, file, ensure_ascii=False, indent=2)
    del valid_parts, validation_export, valid_base
    valid_matrix = encode_prepared(X_valid, schema)
    print('Готово:', written_rows, 'train |', len(X_valid), 'validation |', len(feature_columns), 'признаков')
    print('Размер TSV, GB:', round(TRAIN_TEXT_PATH.stat().st_size / 2**30, 2))

## 4. Экспортируем код применения и фиксируем параметры

Чтобы модель не зависела от Colab, сохраняем `inference.py`: загрузка `model.txt`, схема категорий и функция `predict_prepared`. Её вход — таблица признаков заявок, не сырая история. Для сырой истории рядом лежат `feature_builder.py` и `feature_settings.json`.

Деревья содержат веса и правила разбиений. Нативного `model.txt` вместе со схемой достаточно для прогнозирования. Дополнительно сохраняем joblib-словарь `model_bundle.joblib`; это словарь с Booster и схемой, а не sklearn Pipeline.


In [ ]:
if TRAIN_NEW_MODEL:
    parameters = {
        'objective': 'binary', 'metric': ['auc', 'binary_logloss'],
        'learning_rate': 0.05, 'num_leaves': 31, 'min_data_in_leaf': 100,
        'lambda_l2': 3.0, 'feature_fraction': 0.9,
        'scale_pos_weight': 1.0, 'max_bin': 63, 'min_data_in_bin': 5,
        'force_col_wise': True, 'histogram_pool_size': 256,
        'num_threads': N_THREADS, 'seed': RANDOM_STATE, 'verbosity': -1
    }
    dataset_parameters = {'header': True, 'label_column': 0, 'two_round': True,
                          'max_bin': parameters['max_bin'], 'min_data_in_bin': parameters['min_data_in_bin'],
                          'feature_pre_filter': False, 'verbosity': -1, 'num_threads': N_THREADS}
    metadata = {'model': 'LightGBM full previous train period', 'train_rows': len(train_ids),
        'validation_rows': len(valid_ids), 'parameters': parameters, 'num_boost_round': NUM_BOOST_ROUND,
        'balance': 'none', 'calibrated': False, 'holdout_evaluated': False,
        'split_manifest': str(RUN_DIR / 'split_manifest.json'), 'target': feature_settings['target'],
        'versions': {'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__,
                     'lightgbm': lgb.__version__, 'joblib': joblib.__version__},
        'iteration_choice': '600 fixed before external validation; no early stopping on external validation'}
    with open(MODEL_DIR / 'metadata.json', 'w', encoding='utf-8') as file:
        json.dump(metadata, file, ensure_ascii=False, indent=2)
    shutil.copy2(RUN_DIR / 'split_manifest.json', MODEL_DIR / 'split_manifest.json')

    feature_builder_source = 'import numpy as np\nimport pandas as pd\n'
    feature_builder_source += 'code_columns = ' + repr(code_columns) + '\n'
    feature_builder_source += 'flag_columns = ' + repr(flag_columns) + '\n'
    feature_builder_source += 'payment_columns = ' + repr(payment_columns) + '\n'
    feature_builder_source += "def make_base_features(raw, application_ids, vocabulary):\n    groups = raw.groupby('id', sort=False)\n    index = pd.Index(application_ids, name='id')\n    features = pd.DataFrame(index=index)\n    product_counts = groups.size()\n    features['n_credit_products'] = product_counts.reindex(index, fill_value=0)\n    features['has_history'] = (features.n_credit_products > 0).astype('int8')\n\n    # Для кодов допустимо число различных категорий, но не среднее значение кода.\n    diversity = groups[code_columns].nunique().add_suffix('__nunique')\n    missing_share = raw[code_columns].isna().groupby(raw.id).mean().add_suffix('__missing_share')\n    # Среднее бинарного флага = доля единиц; отдельно учитываем пропуски флагов.\n    flag_share = groups[flag_columns].mean().add_suffix('__share')\n    flag_missing = raw[flag_columns].isna().groupby(raw.id).mean().add_suffix('__missing_share')\n    features = features.join([diversity, missing_share, flag_share, flag_missing])\n\n    frequency_blocks = []\n    for col, categories in vocabulary.items():\n        counts = raw.groupby(['id', col]).size().unstack(col, fill_value=0)\n        counts = counts.reindex(columns=categories, fill_value=0)\n        shares = counts.div(product_counts, axis=0).fillna(0)\n        shares.columns = [f'{col}__code_{value}_share' for value in categories]\n        frequency_blocks.append(shares)\n        unknown = raw[col].notna() & ~raw[col].isin(categories)\n        unknown_share = unknown.groupby(raw.id).mean().rename(f'{col}__unknown_share')\n        frequency_blocks.append(unknown_share.to_frame())\n    features = features.join(frequency_blocks)\n\n    # rn используется для выбора последнего продукта, но не попадает в X.\n    latest = raw.sort_values(['id', 'rn']).drop_duplicates('id', keep='last').set_index('id')\n    for col in code_columns + flag_columns:\n        values = latest[col].reindex(index)\n        features[f'latest__{col}'] = values.map(\n            lambda value: 'missing' if pd.isna(value) else f'code_{value:g}'\n        )\n\n    # Сравниваем статусы на равенство. Номер категории не задаёт тяжесть события.\n    payments = raw[payment_columns]\n    left = payments.iloc[:, :-1].to_numpy()\n    right = payments.iloc[:, 1:].to_numpy()\n    observed = pd.notna(left) & pd.notna(right)\n    comparisons = observed.sum(axis=1)\n    changes = ((left != right) & observed).sum(axis=1)\n    change_share = np.divide(changes, comparisons,\n                             out=np.full(len(raw), np.nan), where=comparisons > 0)\n    changes_by_id = pd.Series(change_share, index=raw.index).groupby(raw.id)\n    features['payment_change_share_mean'] = changes_by_id.mean()\n    features['payment_change_share_max'] = changes_by_id.max()\n    features['payment_pair_observed_share'] = pd.Series(\n        observed.mean(axis=1), index=raw.index).groupby(raw.id).mean()\n\n    numeric_columns = features.select_dtypes(include='number').columns\n    # Ноль допустим для отсутствующих частот; NaN в среднем наблюдавшегося флага\n    # (все значения пропущены) сохраняем для будущего train-only imputer.\n    no_history = features.has_history.eq(0)\n    features.loc[no_history, numeric_columns] = 0\n    share_columns = [col for col in features if col.endswith('_share') and '__code_' in col]\n    features[share_columns] = features[share_columns].fillna(0)\n    features[numeric_columns] = features[numeric_columns].astype('float32')\n    return features\ndef make_application_features(raw, application_ids, vocabulary):\n    features = make_base_features(raw, application_ids, vocabulary)\n    count_features = {}\n    for flag in flag_columns:\n        for value in [0, 1]:\n            count = raw[flag].eq(value).groupby(raw.id).sum()\n            count_features[f'extra__{flag}__count_{value}'] = count.reindex(features.index, fill_value=0)\n    features = features.join(pd.DataFrame(count_features, index=features.index))\n    ordered = raw.sort_values(['id', 'rn'])\n    diversity_fields = [col for col in ['enc_loans_credit_type', 'enc_loans_credit_status',\n        'pre_loans_outstanding', 'pre_util', 'pre_loans3060', 'pre_loans6090', 'pre_loans90'] if col in raw]\n    for k in [3, 5]:\n        recent = ordered.groupby('id', sort=False).tail(k)\n        recent_groups = recent.groupby('id', sort=False)\n        shares = recent_groups[flag_columns].mean().add_prefix(f'extra__last_{k}__').add_suffix('__share')\n        diversity = recent_groups[diversity_fields].nunique().add_prefix(f'extra__last_{k}__').add_suffix('__nunique')\n        features = features.join([shares, diversity])\n    extras = [col for col in features if col.startswith('extra__')]\n    features.loc[features.has_history.eq(0), extras] = 0\n    features[extras] = features[extras].astype('float32')\n    return features\n\n"
    (MODEL_DIR / 'feature_builder.py').write_text(feature_builder_source, encoding='utf-8')

In [ ]:
if TRAIN_NEW_MODEL:
    inference_source = "from pathlib import Path\nimport json\nimport numpy as np\nimport lightgbm as lgb\n\ndef encode_prepared(features, schema):\n    numeric = features[schema['numerical_features']].to_numpy(dtype='float32', copy=True)\n    encoded = np.empty((len(features), len(schema['categorical_features'])), dtype='float32')\n    for i, col in enumerate(schema['categorical_features']):\n        values = features[col].fillna('missing').astype(str)\n        encoded[:, i] = values.map(schema['category_maps'][col]).fillna(-1).to_numpy(dtype='float32')\n    return np.concatenate([numeric, encoded], axis=1)\n\ndef load_model(model_dir):\n    model_dir = Path(model_dir)\n    with open(model_dir / 'schema.json', encoding='utf-8') as file:\n        schema = json.load(file)\n    return lgb.Booster(model_file=str(model_dir / 'model.txt')), schema\n\ndef predict_prepared(features, model_dir):\n    booster, schema = load_model(model_dir)\n    return booster.predict(encode_prepared(features, schema))\n"
    (MODEL_DIR / 'inference.py').write_text(inference_source, encoding='utf-8')

## 5. Загружаем компактный Dataset и обучаем один LightGBM

LightGBM преобразует TSV в интервалы значений и строит деревья на полном train. `two_round=True` поддерживает загрузку большого текстового файла в два прохода. Полностью потоковым обучение не становится: компактный Dataset всё равно хранится в памяти.

Validation выводится в журнале для наблюдения, но не останавливает обучение и не выбирает число деревьев. 600 итераций зафиксированы заранее. На каждом 100-м дереве обновляется checkpoint; после fit сразу сохраняем итог. Если среда отключится, можно применить checkpoint, но его не следует выдавать за завершённую модель.

In [ ]:
if TRAIN_NEW_MODEL:
    train_dataset = lgb.Dataset(str(TRAIN_TEXT_PATH), params=dataset_parameters,
                              feature_name=feature_columns, categorical_feature=schema['categorical_indices'],
                              free_raw_data=True)
    train_dataset.construct()
    assert train_dataset.num_data() == len(train_ids)
    expected_labels = np.concatenate([train_labels.loc[train_ids[train_ids % N_SHARDS == shard]].to_numpy()
                                     for shard in range(N_SHARDS)])
    np.testing.assert_array_equal(train_dataset.get_label(), expected_labels)
    del expected_labels
    valid_dataset = lgb.Dataset(valid_matrix, label=y_valid.to_numpy(), reference=train_dataset,
                              feature_name=feature_columns, categorical_feature=schema['categorical_indices'],
                              free_raw_data=True)
    evaluation_history = {}
    def checkpoint_callback(environment):
        iteration = environment.iteration + 1
        if iteration % CHECKPOINT_EVERY == 0:
            environment.model.save_model(str(MODEL_DIR / 'checkpoint.txt'), num_iteration=iteration)
            with open(MODEL_DIR / 'checkpoint_status.json', 'w', encoding='utf-8') as file:
                json.dump({'completed_iterations': iteration, 'requested_iterations': NUM_BOOST_ROUND,
                           'final_fit_completed': False}, file, indent=2)
    checkpoint_callback.order = 30
    checkpoint_callback.before_iteration = False
    start = time.perf_counter()
    booster = lgb.train(parameters, train_dataset, num_boost_round=NUM_BOOST_ROUND,
                        valid_sets=[valid_dataset], valid_names=['validation_monitor_only'],
                        callbacks=[lgb.log_evaluation(100), lgb.record_evaluation(evaluation_history), checkpoint_callback])
    booster.save_model(str(MODEL_DIR / 'model.txt'))
    joblib.dump({'booster': booster, 'schema': schema}, MODEL_DIR / 'model_bundle.joblib', compress=3)
    metadata['fit_seconds'] = time.perf_counter() - start
    metadata['actual_iterations'] = booster.current_iteration()
    metadata['fit_completed'] = True
    with open(MODEL_DIR / 'metadata.json', 'w', encoding='utf-8') as file:
        json.dump(metadata, file, ensure_ascii=False, indent=2)
    with open(RUN_DIR / 'learning_history.json', 'w', encoding='utf-8') as file:
        json.dump(evaluation_history, file, indent=2)
    print('Обучение завершено, модель сохранена:', MODEL_DIR)
    print('Время, мин:', round(metadata['fit_seconds'] / 60, 1))

    del train_dataset, valid_dataset
    shutil.rmtree(LOCAL_DIR)
    gc.collect()

## 6. Метрики и сравнение с моделями 200k

ROC-AUC основной, AP дополнительный. Brier/log loss показывают качество вероятностей. Политика 80% — диагностическая: порог подбирается по validation. Сравниваем с сохранёнными прогнозами 200k на тех же ID и метках; не обучаем эти модели заново.


In [ ]:
if TRAIN_NEW_MODEL:
    def metrics(labels, probability):
        approved = probability < np.quantile(probability, APPROVAL_TARGET)
        values = np.asarray(labels)
        return {'roc_auc': float(roc_auc_score(labels, probability)),
                'average_precision': float(average_precision_score(labels, probability)),
                'brier_score': float(brier_score_loss(labels, probability)),
                'log_loss': float(log_loss(labels, probability, labels=[0, 1])),
                'approval_rate': float(approved.mean()),
                'default_rate_approved': float(values[approved].mean()) if approved.any() else None,
                'share_defaults_rejected': float(values[~approved].sum() / values.sum())}
    probability = booster.predict(valid_matrix)
    probabilities = {'LightGBM full train period': probability}
    rows = [{'model': 'LightGBM full train period', 'train_rows': len(train_ids), **metrics(y_valid, probability)}]
    pd.DataFrame({'id': valid_ids, 'flag': y_valid.to_numpy(), 'model_probability': probability}).to_parquet(
        MODEL_DIR / 'validation_predictions.parquet', index=False)
    model_paths = {'LightGBM full train period': MODEL_DIR}
    comparison = pd.DataFrame(rows).set_index('model').sort_values(['roc_auc', 'average_precision'], ascending=False)
    display(comparison)
    comparison.to_csv(RUN_DIR / 'comparison_validation.csv')
    with open(MODEL_DIR / 'evaluation.json', 'w', encoding='utf-8') as file:
        json.dump({'validation_metrics': metrics(y_valid, probability), 'holdout_evaluated': False}, file, indent=2)
    best_name = comparison.index[0]
    recommendation = {'model': best_name, 'model_directory': str(model_paths[best_name]),
                      'selection': 'fixed LightGBM configuration; validation evaluation only', 'holdout_evaluated': False}
    with open(RUN_DIR / 'recommended_model.json', 'w', encoding='utf-8') as file:
        json.dump(recommendation, file, ensure_ascii=False, indent=2)
    print('Оценена модель:', best_name)

## 7. Политика одобрения, калибровка и краткий отчёт

Выбранный диагностический порог 80% записываем в JSON. На будущих заявках/holdout его нужно применять без повторного подбора процентиля, если проверяем фиксированную политику. Оптимальный бизнес-порог требует отдельной фиксации экономических предположений.


In [ ]:
if TRAIN_NEW_MODEL:
    policy_rows = []
    y_array = y_valid.to_numpy()
    for name, p in probabilities.items():
        for share in [0.60, 0.70, 0.80, 0.90, 0.95]:
            threshold = float(np.quantile(p, share))
            approved = p < threshold
            policy_rows.append({'model': name, 'approval_target': share, 'threshold': threshold,
                'actual_approval_rate': float(approved.mean()), 'default_rate_approved': float(y_array[approved].mean()) if approved.any() else None,
                'defaults_approved': int(y_array[approved].sum()), 'defaults_rejected': int(y_array[~approved].sum()),
                'good_rejected': int(((y_array == 0) & ~approved).sum())})
    policy_table = pd.DataFrame(policy_rows)
    policy_table.to_csv(RUN_DIR / 'approval_scenarios.csv', index=False)
    display(policy_table)
    fixed_threshold = float(np.quantile(probabilities[best_name], APPROVAL_TARGET))
    with open(RUN_DIR / 'fixed_policy.json', 'w', encoding='utf-8') as file:
        json.dump({'model': best_name, 'threshold': fixed_threshold, 'approval_target_on_validation': APPROVAL_TARGET,
                   'rule': 'approve if score < threshold', 'status': 'diagnostic policy, not economic optimum'}, file, indent=2)
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    calibration_tables = []
    for name, p in probabilities.items():
        RocCurveDisplay.from_predictions(y_valid, p, ax=axes[0], name=name)
        group = policy_table.loc[policy_table.model.eq(name)]
        axes[1].plot(group.actual_approval_rate * 100, group.default_rate_approved * 100, marker='o', label=name)
        bins = pd.qcut(p, 10, duplicates='drop')
        table = pd.DataFrame({'bin': bins, 'score': p, 'flag': y_array}).groupby('bin', observed=True).agg(
            clients=('flag', 'size'), defaults=('flag', 'sum'), predicted=('score', 'mean'), observed=('flag', 'mean')).reset_index()
        table['model'] = name
        if len(table):
            calibration_tables.append(table)
        axes[2].plot(table.predicted, table.observed, marker='o', label=name)
    calibration_table = pd.concat(calibration_tables, ignore_index=True)
    calibration_table.to_csv(RUN_DIR / 'calibration_bins.csv', index=False)
    limit = max(calibration_table.predicted.max(), calibration_table.observed.max()) * 1.1
    axes[0].plot([0, 1], [0, 1], '--', color='gray')
    axes[1].axhline(y_valid.mean() * 100, linestyle='--', color='gray')
    axes[1].set(xlabel='Одобрено, %', ylabel='Дефолты среди одобренных, %', title='Одобрение и риск')
    axes[2].plot([0, limit], [0, limit], '--', color='gray')
    axes[2].set(xlabel='Средний score', ylabel='Доля дефолтов', title='Калибровка')
    for ax in axes:
        ax.legend(fontsize=8)
    plt.tight_layout()
    fig.savefig(RUN_DIR / 'final_summary.png', dpi=150, bbox_inches='tight')
    plt.show()
    report = f'# Итоговое обучение LightGBM\n\nTrain: {len(train_ids)} заявок. Holdout не оценён.\n\n'
    report += f'Рекомендованная модель: {best_name}\n\nПапка: {model_paths[best_name]}\n\n'
    report += '```\n' + comparison.to_string() + '\n```\n\n'
    report += 'Одобрение 80%:\n\n```\n' + policy_table.loc[policy_table.approval_target.eq(0.8)].to_string(index=False) + '\n```\n'
    (RUN_DIR / 'summary.md').write_text(report, encoding='utf-8')

## 8. Проверяем восстановление и показываем готовый вызов


In [ ]:
if TRAIN_NEW_MODEL:
    import importlib.util
    spec = importlib.util.spec_from_file_location('alfa_final_inference', MODEL_DIR / 'inference.py')
    inference = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(inference)
    restored_probability = inference.predict_prepared(X_valid.head(10), MODEL_DIR)
    np.testing.assert_allclose(restored_probability, probability[:10], rtol=1e-10)
    bundle = joblib.load(MODEL_DIR / 'model_bundle.joblib')
    np.testing.assert_allclose(bundle['booster'].predict(inference.encode_prepared(X_valid.head(10), bundle['schema'])), probability[:10], rtol=1e-10)
    display(pd.DataFrame({'id': valid_ids[:10], 'model_probability': restored_probability}))
    print('Восстановление проверено.')
    print('ГОТОВО:', RUN_DIR)
    print('Модель:', MODEL_DIR / 'model.txt')
    print('Код применения:', MODEL_DIR / 'inference.py')

Документация решения с файлами: [LightGBM Dataset](https://lightgbm.readthedocs.io/en/stable/pythonapi/lightgbm.Dataset.html), [two_round и параметры памяти](https://lightgbm.readthedocs.io/en/stable/Parameters.html).